# 📥 Notebook 00 — Environment Setup & Data Download
> **Purpose:** Install dependencies, verify environment, and load LOBSTER sample files.

Run this notebook once before any other notebook in the pipeline.

---

## 0.1  Install dependencies

In [ ]:
# Run once — comment out after first install
import subprocess, sys
pkgs = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'scikit-learn', 'xgboost', 'torch', 'streamlit',
    'tqdm', 'joblib'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('All packages installed.')

## 0.2  Verify imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import torch
import xgboost
import sklearn
import matplotlib

print(f'NumPy      {np.__version__}')
print(f'Pandas     {pd.__version__}')
print(f'PyTorch    {torch.__version__}')
print(f'XGBoost    {xgboost.__version__}')
print(f'Scikit     {sklearn.__version__}')
print(f'Matplotlib {matplotlib.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

## 0.3  LOBSTER data download

Download the sample files from **https://lobsterdata.com/info/DataSamples.php**

Place both files in the `data/` folder:
```
LOBSTER_Capstone/
└── data/
    ├── AAPL_2012-06-21_34200000_57600000_message_10.csv
    └── AAPL_2012-06-21_34200000_57600000_orderbook_10.csv
```

Or update `DATA_CFG` below to point to your own paths.

In [ ]:
import os

# ── Update these paths if needed ─────────────────────────────────
DATA_CFG = {
    'msg_file':  'data/AAPL_2012-06-21_34200000_57600000_message_10.csv',
    'book_file': 'data/AAPL_2012-06-21_34200000_57600000_orderbook_10.csv',
    'ticker':    'AAPL',
    'n_levels':  10,
}

# Save config so later notebooks can import it
import json
os.makedirs('data', exist_ok=True)
with open('data/config.json', 'w') as f:
    json.dump(DATA_CFG, f, indent=2)
print('Config saved to data/config.json')

# Verify files exist
for key, path in [(k, v) for k, v in DATA_CFG.items() if 'file' in k]:
    exists = os.path.isfile(path)
    status = '✅' if exists else '❌ NOT FOUND'
    size   = f'  ({os.path.getsize(path)/1e6:.2f} MB)' if exists else ''
    print(f'  {key}: {path}  {status}{size}')

## 0.4  Quick sanity load

In [ ]:
from utils.lobster_loader import load_message, load_orderbook, merge_files, clean_lobster

msg_df  = load_message(DATA_CFG['msg_file'])
book_df = load_orderbook(DATA_CFG['book_file'])
df      = merge_files(msg_df, book_df)
df_clean = clean_lobster(df)

print('\nFirst 3 rows (message cols):')
display(df_clean[['Time','Type','Size','Price','Direction']].head(3))
print('\nFirst 3 rows (orderbook cols):')
display(df_clean[['AskP1','AskS1','BidP1','BidS1']].head(3))

## 0.5  Save cleaned base DataFrame

In [ ]:
df_clean.to_parquet('data/lobster_clean.parquet', index=False)
print(f'Saved data/lobster_clean.parquet  ({len(df_clean):,} rows)')

---
> ✅ **Setup complete.** Proceed to `01_eda_and_preprocessing.ipynb`.